# Exploratory Data Analysis & Synthetic Data Generation

In this notebook, we will generate a synthetic retail customer dataset to simulate a realistic business environment. We will then perform Exploratory Data Analysis (EDA) to understand the distribution of Customer Lifetime Value (CLV) and the relationships between various customer features.

### Business Context
Our goal is to build an intelligent system that predicts the future value of a customer based on their historical behavior. To do this, we need data. We will simulate 3,000 customers categorized into 4 distinct tiers:
1. **Champions**: High spenders, frequent buyers, very loyal.
2. **Growers**: Recent customers with growing potential.
3. **At-Risk**: Older customers who haven't purchased recently.
4. **Hibernating**: Low value, churned or almost churned customers.

## Step 1: Import Libraries
We begin by importing the necessary Python libraries. We use `numpy` for mathematical operations and random number generation, `pandas` for data manipulation, and plotting libraries for visualization.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os

# Set random seed for reproducibility
np.random.seed(42)

## Step 2: Define Constants
We define global constants for our simulation. We will generate `N = 3000` customers and assume a retail gross margin of 35% for CLV calculations.

In [ ]:
N = 3000
GROSS_MARGIN = 0.35

## Step 3: Define the Data Generation Function
This function creates a subset of customers for a specific CLV tier. It uses probability distributions (like Log-Normal for spending, Uniform for ratios) to ensure the data looks realistic rather than perfectly uniform.

- `tenure_months`: How long they have been a customer.
- `total_orders`: Number of historical orders.
- `total_spend`: Total amount spent (using log-normal distribution to simulate long-tail retail spending).
- `recency_days`: Days since their last purchase.

In [ ]:
def generate_tier(n, tier_name, config):
    data = {
        'tenure_months':     np.random.randint(config['tenure'][0], config['tenure'][1], n),
        'total_orders':      np.random.randint(config['orders'][0], config['orders'][1], n),
        'total_spend':       np.clip(
            np.random.lognormal(mean=np.log(config['spend_mean']),
                                sigma=config['spend_sigma'], size=n),
            config['spend_clip'][0], config['spend_clip'][1]
        ),
        'recency_days':      np.random.randint(config['recency'][0], config['recency'][1], n),
        'age':               np.random.randint(config['age'][0], config['age'][1], n),
        'income_bracket':    np.random.choice(config['income'], size=n),
        'nps_score':         np.random.randint(config['nps'][0], config['nps'][1], n),
        'online_ratio':      np.random.uniform(config['online'][0], config['online'][1], n),
        'return_rate':       np.random.uniform(config['return_rate'][0], config['return_rate'][1], n),
        'support_tickets':   np.random.randint(config['tickets'][0], config['tickets'][1], n),
        'discount_usage':    np.random.uniform(config['discount'][0], config['discount'][1], n),
        'category_preference': np.random.choice(
            config['categories'], size=n,
            p=config.get('cat_probs', None)
        ),
        'region':            np.random.choice(['North', 'South', 'East', 'West'], size=n),
        'gender':            np.random.choice(['M', 'F'], size=n),
        'clv_tier':          tier_name,
    }
    return pd.DataFrame(data)

## Step 4: Configure the 4 Customer Tiers
Here we define the behavioral boundaries for each segment. For example, Champions buy very frequently and spend a lot, whereas Hibernating customers have high recency (haven't bought in a long time) and low spend.

In [ ]:
tier1_config = {
    'tenure': (24, 84), 'orders': (40, 200), 'spend_mean': 8000, 'spend_sigma': 0.4,
    'spend_clip': (2500, 15000), 'recency': (1, 30), 'age': (30, 65), 'income': [3, 4, 4, 4],
    'nps': (8, 11), 'online': (0.3, 0.8), 'return_rate': (0.01, 0.08), 'tickets': (0, 5),
    'discount': (0.0, 0.15), 'categories': ['Electronics', 'Luxury', 'Home'], 'cat_probs': [0.4, 0.4, 0.2]
}
tier2_config = {
    'tenure': (6, 36), 'orders': (12, 60), 'spend_mean': 1500, 'spend_sigma': 0.45,
    'spend_clip': (800, 2500), 'recency': (15, 90), 'age': (25, 55), 'income': [2, 3, 3],
    'nps': (6, 10), 'online': (0.4, 0.9), 'return_rate': (0.05, 0.15), 'tickets': (1, 8),
    'discount': (0.1, 0.35), 'categories': ['Clothing', 'Sports', 'Tech'], 'cat_probs': [0.4, 0.3, 0.3]
}
tier3_config = {
    'tenure': (12, 48), 'orders': (4, 20), 'spend_mean': 450, 'spend_sigma': 0.5,
    'spend_clip': (200, 800), 'recency': (60, 200), 'age': (20, 70), 'income': [1, 2, 2],
    'nps': (4, 8), 'online': (0.2, 0.6), 'return_rate': (0.1, 0.3), 'tickets': (3, 15),
    'discount': (0.3, 0.7), 'categories': ['Grocery', 'Pharmacy', 'Clothing'], 'cat_probs': None
}
tier4_config = {
    'tenure': (1, 24), 'orders': (1, 8), 'spend_mean': 80, 'spend_sigma': 0.6,
    'spend_clip': (10, 200), 'recency': (180, 365), 'age': (18, 75), 'income': [1, 1, 2],
    'nps': (1, 7), 'online': (0.1, 0.5), 'return_rate': (0.2, 0.5), 'tickets': (2, 20),
    'discount': (0.4, 1.0), 'categories': ['Grocery', 'Pharmacy'], 'cat_probs': None
}

## Step 5: Generate and Combine the Data
We generate the 3,000 customers distributed across the 4 tiers, concatenate them into a single dataframe, assign unique IDs, and shuffle the rows.

In [ ]:
df_champions   = generate_tier(600,  'Champions',   tier1_config)
df_growers     = generate_tier(900,  'Growers',     tier2_config)
df_atrisk      = generate_tier(800,  'At-Risk',     tier3_config)
df_hibernating = generate_tier(700,  'Hibernating', tier4_config)

df = pd.concat([df_champions, df_growers, df_atrisk, df_hibernating], ignore_index=True)
df['customer_id'] = [f"CUST_{i+1:05d}" for i in range(len(df))]
df = df.sample(frac=1, random_state=42).reset_index(drop=True)
df.head()

## Step 6: Feature Engineering (RFM Metrics)
RFM stands for Recency, Frequency, and Monetary value. It is a proven marketing model for behavior-based customer segmentation.
- **Recency**: How recently a customer has made a purchase (we divide into 5 quintiles where 5 is best).
- **Frequency**: How often they purchase.
- **Monetary**: How much they spend.
We compute these base metrics and an aggregated `rfm_score`.

In [ ]:
df['avg_order_value'] = (df['total_spend'] / df['total_orders']).clip(lower=1.0)
df['tenure_years'] = df['tenure_months'] / 12.0
df['purchase_frequency'] = (df['total_orders'] / df['tenure_years']).clip(upper=365.0)

# Create 1-5 quintile scores for RFM
df['recency_score'] = pd.qcut(df['recency_days'], q=5, labels=[5, 4, 3, 2, 1], duplicates='drop').astype(int)
df['frequency_score'] = pd.qcut(df['total_orders'], q=5, labels=[1, 2, 3, 4, 5], duplicates='drop').astype(int)
df['monetary_score'] = pd.qcut(df['total_spend'], q=5, labels=[1, 2, 3, 4, 5], duplicates='drop').astype(int)

df['rfm_score'] = df['recency_score'] + df['frequency_score'] + df['monetary_score']

## Step 7: Calculate the Target Variable (12-Month CLV)
To train an ML model, we need a ground truth target variable. We calculate a theoretical CLV using a standard financial formula:
`CLV = (Average Order Value × Purchase Frequency × Gross Margin) / Churn Rate`
We introduce some random log-normal noise to simulate real-world variance.

In [ ]:
# Estimate retention based on RF score
base_retention = (df['recency_score'] + df['frequency_score']) / 10.0
noise = np.random.normal(0, 0.05, size=len(df))
df['retention_rate'] = (base_retention + noise).clip(0.05, 0.95)
df['churn_rate'] = 1.0 - df['retention_rate']

# Calculate final CLV
raw_clv = (df['avg_order_value'] * df['purchase_frequency'] * GROSS_MARGIN) / df['churn_rate']
clv_noise = np.random.lognormal(0, 0.15, size=len(df))
df['clv_12month'] = (raw_clv * clv_noise).clip(5.0, 50000.0).round(2)

# Create categorical segments
df['clv_segment'] = pd.cut(df['clv_12month'], bins=[0, 200, 800, 2500, 50001], labels=['Low', 'Medium', 'High', 'Very High'])

# Save to disk
os.makedirs("../data", exist_ok=True)
df.to_csv("../data/customers.csv", index=False)
print("Data successfully generated and saved to data/customers.csv")

## Step 8: Exploratory Data Analysis (EDA)
Now that we have our dataset, let's visualize the distribution of our target variable, CLV. Because retail spending follows a power law (a few customers spend a lot), we expect a right-skewed distribution.

In [ ]:
plt.figure(figsize=(10,6))
sns.histplot(df['clv_12month'], bins=50, kde=True, color='purple')
plt.title('Distribution of 12-Month CLV')
plt.xlabel('CLV ($)')
plt.ylabel('Number of Customers')
plt.show()

Next, we look at the relationship between historical Total Spend and future 12-Month CLV. We color code the points by their CLV segment. Notice the positive correlation: customers who have historically spent more are predicted to bring more future value.

In [ ]:
plt.figure(figsize=(10,6))
sns.scatterplot(data=df, x='total_spend', y='clv_12month', hue='clv_segment', 
                palette={'Very High': 'orange', 'High': 'green', 'Medium': 'blue', 'Low': 'red'})
plt.title('Total Historical Spend vs Predicted 12-Month CLV')
plt.xlabel('Total Spend ($)')
plt.ylabel('12-Month CLV ($)')
plt.show()